In [ ]:
import IPython.display
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.visualization
import named_arrays as na
import ctis

In [ ]:
velocity = na.linspace(-500, 500, axis="wavelength", num=21) * u.km / u.s

In [ ]:
wavelength_rest = 171 * u.AA

In [ ]:
position_scene = na.Cartesian2dVectorLinearSpace(
    start=-10 * u.arcsec,
    stop=10 * u.arcsec,
    axis=na.Cartesian2dVectorArray("scene_x", "scene_y"),
    num=na.Cartesian2dVectorArray(256, 256),
)

In [ ]:
position_sensor = na.Cartesian2dVectorArray(
    x=na.arange(0, 128 + 1, axis="sensor_x") * u.pix,
    y=na.arange(0, 64 + 1, axis="sensor_y") * u.pix,
)

In [ ]:
coordinates_scene = na.DopplerPositionalVectorArray.from_velocity(
    velocity=velocity,
    wavelength_rest=wavelength_rest,
    position=position_scene,
)

In [ ]:
coordinates_sensor = na.DopplerPositionalVectorArray.from_velocity(
    velocity=velocity,
    wavelength_rest=wavelength_rest,
    position=position_sensor,
)

In [ ]:
scene = ctis.scenes.gaussians(coordinates_scene)

In [ ]:
scene = scene + scene.max() / 100

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=2,
        gridspec_kw=dict(width_ratios=[.9,.1]),
        constrained_layout=True,
    )
    ax, cax = axs
    colorbar = na.plt.rgbmesh(
        C=scene,
        axis_wavelength="wavelength",
        ax=ax,
        vmin=0,
        vmax=scene.outputs.max(),
    )
    na.plt.pcolormesh(
        C=colorbar,
        axis_rgb="wavelength",
        ax=cax,
    )
    ax.set_aspect("equal")
    ax.set_xlabel(f"scene $x$ ({ax.get_xlabel()})")
    ax.set_ylabel(f"scene $y$ ({ax.get_ylabel()})")
    cax.xaxis.set_ticks_position("top")
    cax.xaxis.set_label_position("top")
    cax.yaxis.tick_right()
    cax.yaxis.set_label_position("right")

In [ ]:
spectrum = scene.outputs.mean(("scene_x", "scene_y"))

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    ax2 = ax.twiny()
    na.plt.stairs(
        velocity,
        spectrum,
        ax=ax,
    )
    na.plt.stairs(
        scene.inputs.wavelength,
        spectrum,
        ax=ax2
    )
    ax.set_xlabel(f"Doppler velocity ({ax.get_xlabel()})")
    ax2.set_xlabel(f"wavelength ({ax2.get_xlabel()})")
    ax.set_ylabel(f"average radiance ({ax.get_ylabel()})")

In [ ]:
angle = na.linspace(0, 360, num=4, axis="channel", endpoint=False) * u.deg + 5.64 * u.deg

In [ ]:
dispersion = 10 * u.km / u.s
dispersion = dispersion.to(u.AA, equivalencies=u.doppler_optical(wavelength_rest))
dispersion = (dispersion - wavelength_rest) / u.pix
dispersion.to(u.mAA / u.pix)

In [ ]:
instrument = ctis.instruments.IdealInstrument(
    area_effective=1 * u.cm ** 2,
    timedelta_exposure=20 * u.s,
    plate_scale=.4 * u.arcsec / u.pix,
    dispersion=dispersion,
    angle=angle,
    wavelength_ref=wavelength_rest,
    position_ref=na.Cartesian2dVectorArray(64, 32) * u.pix,
    coordinates_scene=coordinates_scene,
    coordinates_sensor=coordinates_sensor,
    channel="dispersion angle = " + angle.to_string_array("%03d"),
    axis_channel="channel",
    axis_wavelength="wavelength",
    axis_scene_xy=("scene_x", "scene_y"),
    axis_sensor_xy=("sensor_x", "sensor_y"),
)

In [ ]:
%%time
images = instrument.image(scene)

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(
        constrained_layout=True,
        figsize=(9.2, 4),
    )
    norm = plt.Normalize(
        vmin=0,
        vmax=images.outputs.value.ndarray.max(),
    )
    colorizer = plt.Colorizer(
        cmap="gray",
        norm=norm,
    )
    ani = na.plt.pcolormovie(
        instrument.channel,
        images.inputs.position.x,
        images.inputs.position.y,
        C=images.outputs.value,
        axis_time="channel",
        ax=ax,
        kwargs_pcolormesh=dict(
            colorizer=colorizer,
        ),
    )
    plt.colorbar(
        mappable=plt.cm.ScalarMappable(colorizer=colorizer),
        ax=ax,
        label=f"signal ({images.outputs.unit:latex_inline})",
    )
    ax.set_aspect("equal")
    ax.set_xlabel(f"sensor $x$ ({images.inputs.position.x.unit})")
    ax.set_ylabel(f"sensor $y$ ({images.inputs.position.y.unit})")

result = ani.to_jshtml(fps=2)
result = IPython.display.HTML(result)

plt.close(ani._fig)

result

In [ ]:
mart = ctis.inverters.MartInverter(
    instrument=instrument,
    intermediate=True,
)

In [ ]:
inversion = mart(images)

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=3,
        gridspec_kw=dict(width_ratios=[.5, .5, .1]),
        constrained_layout=True,
        figsize=(10, 4.5),
    )
    ax1, ax2, cax = axs
    ax2.set_yticklabels([])
    na.plt.rgbmesh(
        C=scene,
        axis_wavelength="wavelength",
        ax=ax1,
        vmin=0,
        vmax=scene.outputs.max(),
    )
    label = "iteration = " + inversion.iteration.to_string_array("%d") + "\n"
    name = r"$\langle \chi^2 \rangle$"
    label = label + f"{name} = " + inversion.mean_chi_squared.mean(instrument.axis_channel).to_string_array()
    ani, colorbar = na.plt.rgbmovie(
        label,
        scene.inputs.wavelength,
        scene.inputs.position.x,
        scene.inputs.position.y,
        C=inversion.solutions.outputs,
        axis_time=inversion.inverter.axis_iteration,
        axis_wavelength="wavelength",
        ax=ax2,
        vmin=0,
        vmax=scene.outputs.max(),
    )
    na.plt.pcolormesh(
        C=colorbar,
        axis_rgb="wavelength",
        ax=cax,
    )
    ax1.set_title("original")
    ax2.set_title("reconstructed")
    unit_x = scene.inputs.position.x.unit
    unit_y = scene.inputs.position.y.unit
    ax1.set_xlabel(f"scene $x$ ({unit_x:latex_inline})")
    ax2.set_xlabel(f"scene $x$ ({unit_x:latex_inline})")
    ax1.set_ylabel(f"scene $y$ ({unit_y:latex_inline})")
    cax.xaxis.set_ticks_position("top")
    cax.xaxis.set_label_position("top")
    cax.yaxis.tick_right()
    cax.yaxis.set_label_position("right")

result = ani.to_jshtml(fps=10)
result = IPython.display.HTML(result)

plt.close(ani._fig)

result

In [ ]:
solution = inversion.solution

In [ ]:
spectrum_inverted = solution.outputs.mean(("scene_x", "scene_y"))

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    na.plt.stairs(
        scene.inputs.wavelength,
        spectrum,
        ax=ax,
        label="original",
    )
    na.plt.stairs(
        scene.inputs.wavelength,
        spectrum_inverted,
        ax=ax,
        label="reconstructed",
    )
    ax.set_xlabel(f"wavelength ({ax.get_xlabel()})")
    ax2.set_xlabel(f"wavelength ({ax2.get_xlabel()})")
    ax.set_ylabel(f"average radiance ({ax.get_ylabel()})")
    ax.legend()

In [ ]:
inversion.plot_moments(scene);

In [ ]:
fig, ax = plt.subplots(
    nrows=2,
    sharex=True,
    constrained_layout=True,
)
na.plt.plot(
    inversion.iteration,
    inversion.mean_chi_squared,
    ax=ax[0],
    axis=inversion.inverter.axis_iteration,
    label=instrument.channel,
)
na.plt.plot(
    inversion.iteration,
    inversion.correlation_residual,
    ax=ax[1],
    axis=inversion.inverter.axis_iteration,
    label=instrument.channel,
)
ax[0].set_ylabel(r"$\langle \chi^2 \rangle$")
ax[1].set_xlabel("iteration")
ax[1].set_ylabel("signal-correlated residual")
ax[0].set_yscale("log")
ax[0].legend();